# **1. Mounting Google Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **2. Install Dependencies for PySpark**

I installed Java, PySpark, and findspark to set up Spark in this fresh Colab session.

In [2]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install -q -U "pyspark[connect]~=4.0.0" findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


# **3. SparkSession Initialisation**

I initialised the SparkSession with the same settings used in the earlier notebooks, since this notebook needs to process the full dataset again for accurate raw-value aggregations.

In [3]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

import findspark
findspark.init()

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import os, subprocess, glob, shutil

spark = (
    SparkSession.builder
    .appName("AmazonAutomotiveTableauExport")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version : {spark.version}")

Spark version : 4.0.4


# **4. Loading Raw Data and Rebuilding Analysis Columns**

I downloaded the raw Automotive dataset again and rebuilt sentiment, review length, review year, and verified purchase as plain numeric columns. I did this instead of reading from my saved Parquet file, since that file only keeps the scaled features vector from the TF-IDF pipeline, not the original character counts. Exporting from the scaled vector would show small standardized numbers in Tableau instead of real review lengths.

In [4]:
output_dir = "/content/drive/MyDrive/amazon_automotive_sentiment"
tableau_dir = f"{output_dir}/tableau_exports"
os.makedirs(tableau_dir, exist_ok=True)

data_file = "/content/Automotive.jsonl"

check = subprocess.run(["test", "-f", data_file], capture_output=True)
if check.returncode != 0:
    print("Downloading dataset...")
    subprocess.run([
        "wget", "-q",
        "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Automotive.jsonl",
        "-O", data_file
    ], check=True)

df = spark.read.json(data_file).drop("images")

df = (df
    .withColumn("sentiment", F.when(F.col("rating") >= 4.0, 1).otherwise(0))
    .withColumn("review_length", F.length(F.col("text")))
    .withColumn("review_year", (F.col("timestamp") / 1000 / 86400 / 365 + 1970).cast("int"))
    .withColumn("verified_int", F.col("verified_purchase").cast("int"))
)

print(f"Rows loaded : {df.count():,}")

def save_csv(sdf, name):
    sdf.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{tableau_dir}/{name}")
    print(f"Saved {name}")

Rows loaded : 19,955,450


# **5. Dashboard 1 Data — Data Quality and Pipeline Monitoring**

I exported how many rows survived each cleaning stage in the preprocessing notebook, the final sentiment class balance, the proportion of verified purchases, and review volume per year. Together these give Tableau a full picture of whether the cleaning pipeline kept the data healthy and how the dataset is structured over time.

In [5]:
pipeline_stages = spark.createDataFrame([
    ("1. Raw Dataset", 19955450),
    ("2. After Duplicate Removal", 19723226),
    ("3. After Label Creation", 19723226),
    ("4. Final Train + Test", 19707232),
], ["stage", "row_count"])

class_distribution = (df.groupBy("sentiment").count()
    .withColumnRenamed("count", "review_count")
    .orderBy("sentiment"))

verified_split = (df.groupBy("verified_int").count()
    .withColumnRenamed("count", "review_count")
    .orderBy("verified_int"))

volume_by_year = (df.groupBy("review_year").count()
    .withColumnRenamed("count", "review_count")
    .orderBy("review_year"))

save_csv(pipeline_stages, "d1_pipeline_stages")
save_csv(class_distribution, "d1_class_distribution")
save_csv(verified_split, "d1_verified_split")
save_csv(volume_by_year, "d1_volume_by_year")

Saved d1_pipeline_stages
Saved d1_class_distribution
Saved d1_verified_split
Saved d1_volume_by_year


# **6. Dashboard 2 Data — Model Performance and Feature Importance**

I wrote out the final metrics table and the top feature importances I already computed in the training and evaluation notebooks, rather than retraining or reloading the models again here. This dashboard compares how the four algorithms performed and which features drove their decisions.

In [6]:
model_metrics = spark.createDataFrame([
    ("Logistic Regression", 0.8697, 0.8644, 0.8697, 0.8576, 0.9200),
    ("Random Forest", 0.7875, 0.6202, 0.7875, 0.6939, 0.5705),
    ("Gradient Boosted Trees", 0.8113, 0.8270, 0.8113, 0.7502, 0.7831),
    ("Naive Bayes", 0.7579, 0.8097, 0.7579, 0.7741, 0.6407),
], ["model", "accuracy", "precision", "recall", "f1", "auc"])

feature_importance = spark.createDataFrame([
    ("feature_65262", 0.2437, "Logistic Regression"),
    ("feature_12710", 0.1835, "Logistic Regression"),
    ("feature_36229", 0.1744, "Logistic Regression"),
    ("feature_30469", -0.1640, "Logistic Regression"),
    ("feature_31448", 0.1511, "Logistic Regression"),
    ("feature_16706", -0.1502, "Logistic Regression"),
    ("feature_13692", 0.1463, "Gradient Boosted Trees"),
    ("feature_65067", 0.1210, "Gradient Boosted Trees"),
    ("feature_36229", 0.1120, "Gradient Boosted Trees"),
    ("feature_31448", 0.1075, "Gradient Boosted Trees"),
    ("feature_16706", 0.0958, "Gradient Boosted Trees"),
    ("feature_65536", 0.0445, "Gradient Boosted Trees"),
], ["feature", "importance", "model"])

save_csv(model_metrics, "d2_model_metrics")
save_csv(feature_importance, "d2_feature_importance")

Saved d2_model_metrics
Saved d2_feature_importance


# **7. Dashboard 2 Data — Feature Importance for Random Forest and Naive Bayes**

I loaded the Random Forest and Naive Bayes models saved by the training notebook to extract their feature importances, completing the picture alongside Logistic Regression and GBT.

In [7]:
from pyspark.ml import PipelineModel
import numpy as np

models_dir = f"{output_dir}/models"

forest_model = PipelineModel.load(f"{models_dir}/random_forest")
nbayes_model = PipelineModel.load(f"{models_dir}/naive_bayes")

# Random Forest feature importances
forest_classifier = forest_model.stages[-1]
forest_importances = forest_classifier.featureImportances.toArray()
top_forest_idx = np.argsort(-forest_importances)[:6]

forest_importance_rows = [
    (f"feature_{i}", float(forest_importances[i]), "Random Forest")
    for i in top_forest_idx
]

# Naive Bayes — derive importance from theta (log-probability differences between classes)
nbayes_classifier = nbayes_model.stages[-1]
theta = nbayes_classifier.theta.toArray()  # shape: (numClasses, numFeatures)
nbayes_importance = np.abs(theta[1] - theta[0])  # difference between Positive and Negative class log-probs
top_nbayes_idx = np.argsort(-nbayes_importance)[:6]

nbayes_importance_rows = [
    (f"feature_{i}", float(nbayes_importance[i]), "Naive Bayes")
    for i in top_nbayes_idx
]

all_new_importance = spark.createDataFrame(
    forest_importance_rows + nbayes_importance_rows,
    ["feature", "importance", "model"]
)

save_csv(all_new_importance, "d2_feature_importance_extra")

Saved d2_feature_importance_extra


# **8. Dashboard 3 Data — Business Insights**

I calculated the star rating distribution, average review length and helpful votes by sentiment, sentiment trend over time, and a business-friendly breakdown of where the best model's predictions went right and wrong. Together these translate raw model output into a story a non-technical reader could follow: how ratings spread across the scale, how Negative and Positive reviews differ in character, whether sentiment has shifted year on year, and how often the model confuses one class for the other.

In [8]:
rating_distribution = (df.groupBy("rating").count()
    .withColumnRenamed("count", "review_count")
    .orderBy("rating"))

review_patterns = (df.groupBy("sentiment")
    .agg(
        F.round(F.avg("review_length"), 1).alias("avg_review_length"),
        F.round(F.avg("helpful_vote"), 2).alias("avg_helpful_vote"),
        F.count("*").alias("review_count")
    ))

sentiment_trend = (df.groupBy("review_year")
    .agg(
        F.round(F.avg("sentiment"), 3).alias("positive_share"),
        F.count("*").alias("review_count")
    )
    .orderBy("review_year"))

confusion_business = spark.createDataFrame([
    ("Correctly identified Positive", 12044),
    ("Correctly identified Negative", 1687),
    ("Negative reviews missed (predicted Positive)", 1668),
    ("Positive reviews misflagged (predicted Negative)", 389),
], ["category", "count"])

save_csv(rating_distribution, "d3_rating_distribution")
save_csv(review_patterns, "d3_review_patterns")
save_csv(sentiment_trend, "d3_sentiment_trend")
save_csv(confusion_business, "d3_confusion_business")

Saved d3_rating_distribution
Saved d3_review_patterns
Saved d3_sentiment_trend
Saved d3_confusion_business


# **9. Dashboard 4 Data — Scalability and Cost Analysis**

I brought together the training time, resource configuration, and noise-stability results I'd already measured across the training, tuning and evaluation notebooks since this dashboard is about infrastructure decisions and model robustness rather than anything new I needed to calculate here.

In [9]:
training_scalability = spark.createDataFrame([
    ("Logistic Regression", 279.19, "regParam=0.1"),
    ("Random Forest", 678.4, "numTrees=10"),
    ("Gradient Boosted Trees", 969.6, "stepSize=0.2"),
    ("Naive Bayes", 165.9, "smoothing=1.0"),
], ["model", "training_time_seconds", "best_hyperparameter"])

resource_config = spark.createDataFrame([
    ("driver.memory", "8g", "Handles 8.3GB dataset and 65,541-dim sparse vectors"),
    ("shuffle.partitions", "200", "Evenly distributes high-dimensional data across tasks"),
    ("master", "local[*]", "Uses all available CPU cores for parallel processing"),
], ["setting", "value", "reason"])

stability_results = spark.createDataFrame([
    ("Logistic Regression", 0.007, 0.002),
    ("Random Forest", 0.004, 0.000),
    ("Naive Bayes", 0.001, 0.000),
    ("Gradient Boosted Trees", 0.071, 0.007),
], ["model", "auc_drop", "acc_drop"])

save_csv(training_scalability, "d4_training_scalability")
save_csv(resource_config, "d4_resource_config")
save_csv(stability_results, "d4_stability")

Saved d4_training_scalability
Saved d4_resource_config
Saved d4_stability


# **10. Flattening CSV Files for Tableau**

Spark writes each CSV inside its own folder with an auto-generated part-file name, which Tableau doesn't read cleanly. I copied each part-file out into a single flat folder with a proper file name, so they're easy to upload directly to Tableau Public.

In [10]:
flat_dir = f"{output_dir}/tableau_csv_files"
os.makedirs(flat_dir, exist_ok=True)

for folder in glob.glob(f"{tableau_dir}/*"):
    name = os.path.basename(folder)
    part = glob.glob(f"{folder}/part-*.csv")
    if part:
        shutil.copy(part[0], f"{flat_dir}/{name}.csv")
        print(f"Created {name}.csv")

print(f"\nAll CSV files ready in: {flat_dir}")

Created d1_pipeline_stages.csv
Created d1_class_distribution.csv
Created d1_verified_split.csv
Created d1_volume_by_year.csv
Created d2_model_metrics.csv
Created d2_feature_importance.csv
Created d2_feature_importance_extra.csv
Created d3_rating_distribution.csv
Created d3_review_patterns.csv
Created d3_sentiment_trend.csv
Created d3_confusion_business.csv
Created d4_training_scalability.csv
Created d4_resource_config.csv
Created d4_stability.csv

All CSV files ready in: /content/drive/MyDrive/amazon_automotive_sentiment/tableau_csv_files


# **11. Verifying Exported Files**

In [11]:
print("EXPORTED CSV FILES FOR TABLEAU")
print("=" * 50)
for f in sorted(glob.glob(f"{flat_dir}/*.csv")):
    size_kb = os.path.getsize(f) / 1024
    print(f"{os.path.basename(f):<32}{size_kb:>8.1f} KB")

EXPORTED CSV FILES FOR TABLEAU
d1_class_distribution.csv            0.0 KB
d1_pipeline_stages.csv               0.1 KB
d1_verified_split.csv                0.0 KB
d1_volume_by_year.csv                0.3 KB
d2_feature_importance.csv            0.5 KB
d2_feature_importance_extra.csv      0.6 KB
d2_model_metrics.csv                 0.2 KB
d3_confusion_business.csv            0.2 KB
d3_rating_distribution.csv           0.1 KB
d3_review_patterns.csv               0.1 KB
d3_sentiment_trend.csv               0.5 KB
d4_resource_config.csv               0.2 KB
d4_stability.csv                     0.1 KB
d4_training_scalability.csv          0.2 KB


# **12. Spark Session Termination**

I terminated the active SparkSession to release allocated memory and cluster resources.

In [12]:
spark.stop()
print("Spark session stopped successfully.")

Spark session stopped successfully.
